# Player Subsets Comparison

This notebook allows you to compare aggregated statistics between two subsets of NHL players.

## Get players stats for two subsets

In [1]:
#! /home/db/.pyenv/shims/python

import requests
import pandas as pd
from ipywidgets import widgets, VBox, HBox, interactive_output
from IPython.display import display, HTML
import json
from players_common import get_player_data, get_player_info, get_table_with_players_subset_data

# Base server URL to get nhl stats
BASE_URL = "http://localhost:8080"

# Dictionary to store players data for two subsets
subset_a_players = {}  # {player_id: {'name': str, 'data': dict}}
subset_b_players = {}  # {player_id: {'name': str, 'data': dict}}

# Lists to store player search pattern widgets for each subset
subset_a_search_widgets = []
subset_b_search_widgets = []

# UI Components
subset_a_output = widgets.Output()
subset_b_output = widgets.Output()
subset_a_container = VBox()
subset_b_container = VBox()

def create_player_search_widget(player_num):
    """Create a player search pattern widget"""
    return widgets.Text(
        description=f'Player {player_num}:',
        style={'description_width': '100px'}
    )

def add_player_search_widget(subset_name):
    """Add a new player search pattern widget to specified subset"""
    if subset_name == 'a':
        player_num = len(subset_a_search_widgets) + 1
        new_widget = create_player_search_widget(player_num)
        subset_a_search_widgets.append(new_widget)
        update_subset_container('a')
    else:
        player_num = len(subset_b_search_widgets) + 1
        new_widget = create_player_search_widget(player_num)
        subset_b_search_widgets.append(new_widget)
        update_subset_container('b')

def remove_player_search_widget(subset_name, index):
    """Remove a player search pattern widget by index from specified subset"""
    if subset_name == 'a':
        if index < len(subset_a_search_widgets):
            subset_a_search_widgets.pop(index)
            update_subset_container('a')
    else:
        if index < len(subset_b_search_widgets):
            subset_b_search_widgets.pop(index)
            update_subset_container('b')

def update_subset_container(subset_name):
    """Update the container with current player search widgets"""
    widgets_list = subset_a_search_widgets if subset_name == 'a' else subset_b_search_widgets
    container = subset_a_container if subset_name == 'a' else subset_b_container
    
    children = []
    for i, widget in enumerate(widgets_list):
        if i == 0:
            # Add button for first widget
            add_btn = widgets.Button(description='+', button_style='info', width='50px')
            add_btn.on_click(lambda _, s=subset_name: add_player_search_widget(s))
            children.append(HBox([widget, add_btn]))
        else:
            # Add remove button for other widgets
            remove_btn = widgets.Button(description='−', button_style='danger', width='50px')
            remove_btn.on_click(lambda _, s=subset_name, idx=i: remove_player_search_widget(s, idx))
            children.append(HBox([widget, remove_btn]))
    
    container.children = children

def search_subset_players(subset_name):
    """Search and load players for a specific subset"""
    global subset_a_players, subset_b_players
    
    subset_players = subset_a_players if subset_name == 'a' else subset_b_players
    widgets_list = subset_a_search_widgets if subset_name == 'a' else subset_b_search_widgets
    output = subset_a_output if subset_name == 'a' else subset_b_output
    
    with output:
        output.clear_output()
        subset_players.clear()
        
        try:
            for i, widget in enumerate(widgets_list):
                search_pattern = widget.value.strip()
                if search_pattern:
                    print(f"Searching for Player {i+1}: {search_pattern}")
                    player_info = get_player_info(search_pattern)
                    if player_info:
                        player_id, player_name = player_info
                        player_data = get_player_data(player_id)
                        subset_players[player_id] = {
                            'name': player_name,
                            'data': player_data
                        }
                        print(f"  ✓ Data retrieved for {player_name}\n")
                    else:
                        print(f"  ✗ Could not retrieve data for Player {i+1}\n")
                else:
                    print(f"Player {i+1}: Empty search pattern")
                    
            if subset_players:
                print(f"\n✓ Successfully loaded {len(subset_players)} player(s) for Subset {subset_name.upper()}")
        except Exception as e:
            print(f"Error: {str(e)}")

# Initialize with two player search widgets per subset
add_player_search_widget('a')
add_player_search_widget('a')
add_player_search_widget('b')
add_player_search_widget('b')

search_subset_a_button = widgets.Button(
    description='Search Subset A',
    button_style='info'
)
search_subset_b_button = widgets.Button(
    description='Search Subset B',
    button_style='warning'
)

search_subset_a_button.on_click(lambda _: search_subset_players('a'))
search_subset_b_button.on_click(lambda _: search_subset_players('b'))

display(
    VBox([
        widgets.HTML("<b>Subset A</b>"),
        subset_a_container,
        search_subset_a_button,
        subset_a_output,
        
        widgets.HTML("<b>Subset B</b>"),
        subset_b_container,
        search_subset_b_button,
        subset_b_output,
    ])
)

## Season Totals Comparison

In [ ]:
# Season Totals Comparison for Two Subsets

season = widgets.Dropdown(
    options=[],
    description='Season:',
    style={'description_width': '150px'}
)

game_type_id = widgets.Dropdown(
    options=[("Regular Season", 2), ("Playoffs", 3)],
    value=2,
    description='Game Type:',
    style={'description_width': '150px'}
)

display_stats_by_season_button = widgets.Button(
    description='Stats by season',
    button_style='success',
    style={'description_width': '300px'}
)

display_agg_season_totals_button = widgets.Button(
    description='For All Seasons',
    button_style='success',
    style={'description_width': '300px'}
)

def update_season_dropdowns():
    """
    Update season dropdowns by collecting seasons from both subsets;
    Then sort the seasons and select the most recent season by default.
    """
    seasons = set()
    
    for player_info in subset_a_players.values():
        player_data = player_info.get('data')
        if player_data and 'seasonTotals' in player_data:
            seasons.update([s['season'] for s in player_data['seasonTotals'] if s['season']])
    
    for player_info in subset_b_players.values():
        player_data = player_info.get('data')
        if player_data and 'seasonTotals' in player_data:
            seasons.update([s['season'] for s in player_data['seasonTotals'] if s['season']])
    
    sorted_seasons = sorted(seasons)
    season.options = [(str(s), s) for s in sorted_seasons]
    if sorted_seasons:
        # Select most recent season
        season.value = sorted_seasons[-1]

def get_subset_player_names(subset_players):
    """Get concatenated list of player names from a subset"""
    if not subset_players:
        return "Empty"
    names = [player_info['name'] for player_info in subset_players.values()]
    return ', '.join(names)

def highlight_larger(row):
    """Highlight the larger value between the two subsets with red color"""
    styles = [''] * len(row)
    
    try:
        # Get all column names except 'Metric'
        columns = [col for col in row.index if col != 'Metric']
        
        if len(columns) == 2:
            a_val = pd.to_numeric(row[columns[0]], errors='coerce')
            b_val = pd.to_numeric(row[columns[1]], errors='coerce')
            
            if pd.notna(a_val) and pd.notna(b_val):
                a_idx = list(row.index).index(columns[0])
                b_idx = list(row.index).index(columns[1])
                
                if a_val > b_val:
                    styles[a_idx] = 'background-color: red; color: white; font-weight: bold'
                elif b_val > a_val:
                    styles[b_idx] = 'background-color: red; color: white; font-weight: bold'
    except Exception:
        pass
    
    return styles

def stats_by_season():
    """Display season stats for both subsets in a comparison table"""
    def get_subset_season_stats(subset_players, season_val, game_type_val):
        """Aggregate season stats for all players in a subset"""
        if not subset_players:
            return None
        
        aggregated = {}
        numeric_fields = ['gamesPlayed', 'goals', 'assists', 'points', 'pim', 
                        'powerPlayGoals', 'shorthandedGoals', 'gameWinningGoals', 'shots']
        
        for field in numeric_fields:
            aggregated[field] = 0
        
        found_any = False
        for player_info in subset_players.values():
            player_data = player_info.get('data')
            if not player_data or 'seasonTotals' not in player_data:
                continue
            
            # Find matching stats
            matching_stats = [s for s in player_data['seasonTotals'] 
                            if s['season'] == season_val and s['gameTypeId'] == game_type_val and s['leagueAbbrev'] == 'NHL']
            
            if matching_stats:
                found_any = True
                # Aggregate multiple matching stats if needed
                for stat in matching_stats:
                    for field in numeric_fields:
                        aggregated[field] += stat.get(field, 0)
        
        return aggregated if found_any else None

    def display_season_totals():
        if not subset_a_players and not subset_b_players:
            print("Please search for players in both subsets first")
            return
        
        # Get player names for column headers
        subset_a_names = get_subset_player_names(subset_a_players)
        subset_b_names = get_subset_player_names(subset_b_players)
        
        # Get aggregated stats for both subsets
        subset_a_stats = get_subset_season_stats(subset_a_players, season.value, game_type_id.value)
        subset_b_stats = get_subset_season_stats(subset_b_players, season.value, game_type_id.value)
        
        if not subset_a_stats and not subset_b_stats:
            print(f"No season stats found for season {season.value} with selected game type")
            return
        
        # Create comparison table
        comparison_data = []
        stat_fields = ['gamesPlayed', 'goals', 'assists', 'points', 'pim', 
                      'powerPlayGoals', 'shorthandedGoals', 'gameWinningGoals', 'shots']
        
        for field in stat_fields:
            row = {'Metric': field}
            if subset_a_stats:
                row[subset_a_names] = subset_a_stats.get(field, 0)
            else:
                row[subset_a_names] = '-'
            if subset_b_stats:
                row[subset_b_names] = subset_b_stats.get(field, 0)
            else:
                row[subset_b_names] = '-'
            comparison_data.append(row)
        
        df_temp = pd.DataFrame(comparison_data)
        styled = df_temp.style.apply(highlight_larger, axis=1)
        display(styled)
    
    display_season_totals()

def display_career_totals():
    """Display aggregated career totals for both subsets"""
    
    stat_fields = ['gamesPlayed', 'goals', 'assists', 'points', 'plusMinus', 'pim',
                  'powerPlayGoals', 'shorthandedGoals', 'gameWinningGoals', 'shots', 'shootingPctg']
    
    if not subset_a_players and not subset_b_players:
        print("Please search for players in both subsets first.")
        return
    
    def get_subset_career_totals(subset_players):
        """Aggregate career totals for all players in a subset"""
        if not subset_players:
            return None, None
        
        reg_season_agg = {}
        playoff_agg = {}
        
        for field in stat_fields:
            reg_season_agg[field] = 0
            playoff_agg[field] = 0
        
        for player_info in subset_players.values():
            player_data = player_info.get('data')
            if not player_data or 'careerTotals' not in player_data:
                continue
            
            career_totals = player_data['careerTotals']
            
            # Aggregate regular season
            if 'regularSeason' in career_totals:
                reg_season = career_totals['regularSeason']
                for field in stat_fields:
                    reg_season_agg[field] += reg_season.get(field, 0)
            
            # Aggregate playoffs
            if 'playoffs' in career_totals:
                playoff = career_totals['playoffs']
                for field in stat_fields:
                    playoff_agg[field] += playoff.get(field, 0)
        
        has_reg_season = any(v > 0 for v in reg_season_agg.values())
        has_playoffs = any(v > 0 for v in playoff_agg.values())
        
        return (reg_season_agg if has_reg_season else None, playoff_agg if has_playoffs else None)
    
    # Get player names for column headers
    subset_a_names = get_subset_player_names(subset_a_players)
    subset_b_names = get_subset_player_names(subset_b_players)
    
    # Get aggregated career totals
    subset_a_reg, subset_a_playoff = get_subset_career_totals(subset_a_players)
    subset_b_reg, subset_b_playoff = get_subset_career_totals(subset_b_players)
    
    # Display Regular Season Career Totals
    print("\nREGULAR SEASON CAREER TOTALS")
    print("=" * 80)
    
    reg_season_data = get_table_with_players_subset_data(
        fields=stat_fields,
        subset_a=subset_a_reg,
        subset_a_names=subset_a_names,
        subset_b=subset_b_reg,
        subset_b_names=subset_b_names,
    )
    
    if reg_season_data:
        df_reg = pd.DataFrame(reg_season_data)
        styled = df_reg.style.apply(highlight_larger, axis=1)
        display(styled)
    else:
        print("No career regular season totals found.")
    
    # Display Playoffs Career Totals
    print("\n\nPLAYOFFS CAREER TOTALS")
    print("=" * 80)
    
    playoff_data = get_table_with_players_subset_data(
        fields=stat_fields,
        subset_a=subset_a_playoff,
        subset_a_names=subset_a_names,
        subset_b=subset_b_playoff,
        subset_b_names=subset_b_names,
    )
    
    if playoff_data:
        df_playoff = pd.DataFrame(playoff_data)
        styled = df_playoff.style.apply(highlight_larger, axis=1)
        display(styled)
    else:
        print("No career playoff totals found.")

season_output = widgets.Output()
agg_season_totals_output = widgets.Output()

def on_display_season_click(_):
    season_output.clear_output()
    with season_output:
        stats_by_season()
        
def on_display_agg_season_totals_click(b):
    agg_season_totals_output.clear_output()
    with agg_season_totals_output:
        display_career_totals()

display_stats_by_season_button.on_click(on_display_season_click)
display_agg_season_totals_button.on_click(on_display_agg_season_totals_click)

# Update season dropdowns when this cell is first run
update_season_dropdowns()

display(
    VBox([
        HBox([season, game_type_id]),
        display_stats_by_season_button,
        season_output,
        display_agg_season_totals_button,
        agg_season_totals_output
    ])
)